# SARIMAX - Otimizacao automatica das ordens
**Grupo 1 | Bitcoin diário**

Este notebook usa `pmdarima.arima.auto_arima` para selecionar `p,d,q` e `P,D,Q` somente com os 70% iniciais do treino. A sazonalidade semanal é definida como `m=7` (a função otimiza as ordens, não escolhe o período `m`). A seleção minimiza AIC; AIC e BIC do modelo escolhido são exibidos. As 51 features são passadas como variáveis exógenas, padronizadas com estatísticas do treino. O teste não participa da seleção.

O intercepto é permitido durante a busca e está presente no modelo selecionado. A especificação correspondente será reutilizada nos notebooks de walk-forward e diagnóstico. Nenhum arquivo auxiliar é criado.

In [1]:
# SARIMAX - 01: Otimizacao de hiperparametros (Grupo 1)
# Selecao temporal usando apenas os 70% iniciais; o teste fica intocado.
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
TRAIN_RATIO = 0.70

plt.rcParams.update({
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444466',
    'axes.labelcolor': '#ccccee',
    'xtick.color': '#aaaacc',
    'ytick.color': '#aaaacc',
    'text.color': '#e0e0ff',
    'grid.color': '#2a2a4a',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'font.size': 11
})

cwd = Path.cwd().resolve()
root_candidates = list(cwd.parents) + [cwd]
root_candidates += [parent / 'analyses' / 'grupo1' for parent in [cwd, *cwd.parents]]
GROUP_ROOT = next(
    (candidate for candidate in dict.fromkeys(root_candidates)
     if (candidate / '01_dados' / 'feature_metadata.json').is_file()),
    None
)
if GROUP_ROOT is None:
    raise FileNotFoundError('Nao foi possivel localizar analyses/grupo1/01_dados/feature_metadata.json.')

DATA_DIR = GROUP_ROOT / '01_dados'
TRAIN_PATH = DATA_DIR / 'bitcoin_features_train.csv'
TEST_PATH = DATA_DIR / 'bitcoin_features_test.csv'
METADATA_PATH = DATA_DIR / 'feature_metadata.json'

with METADATA_PATH.open('r', encoding='utf-8') as file:
    metadata = json.load(file)

FEATURE_COLS = metadata['feature_cols']
TARGET_COL = metadata['target_col']
train = pd.read_csv(TRAIN_PATH, parse_dates=['date']).sort_values('date').set_index('date')
test = pd.read_csv(TEST_PATH, parse_dates=['date']).sort_values('date').set_index('date')
X_train = train[FEATURE_COLS].astype(float)
y_train = train[TARGET_COL].astype(float)
X_test = test[FEATURE_COLS].astype(float)
y_test = test[TARGET_COL].astype(float)

if not np.isfinite(X_train.to_numpy()).all() or not np.isfinite(y_train.to_numpy()).all():
    raise ValueError('O conjunto de treino contem valores ausentes ou nao finitos.')
if not np.isfinite(X_test.to_numpy()).all() or not np.isfinite(y_test.to_numpy()).all():
    raise ValueError('O conjunto de teste contem valores ausentes ou nao finitos.')

print(f'Raiz do Grupo 1: {GROUP_ROOT}')
print(f'Treino: {len(train)} observacoes | {train.index.min().date()} a {train.index.max().date()}')
print(f'Teste: {len(test)} observacoes | {test.index.min().date()} a {test.index.max().date()}')
print(f'Features exogenas: {len(FEATURE_COLS)} | alvo: {TARGET_COL}')
print(f'Random state: {RANDOM_STATE} | divisao: {metadata["train_ratio"]:.0%}/{1 - metadata["train_ratio"]:.0%}')

Raiz do Grupo 1: C:\Users\raqueltolomei-ieg\OneDrive - Instituto Germinare\Área de Trabalho\3 ano\Séries Temporais\PLSRegretion\pls-regration-comparation\analyses\grupo1
Treino: 1996 observacoes | 2018-11-20 a 2024-05-07
Teste: 856 observacoes | 2024-05-08 a 2026-09-10
Features exogenas: 51 | alvo: target
Random state: 42 | divisao: 70%/30%


In [2]:
from pmdarima.arima import auto_arima

# Padronizacao das exogenas somente com o conjunto de treino.
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    index=X_train.index,
    columns=FEATURE_COLS
)

# Busca stepwise de p,d,q e P,D,Q; m=7 representa sazonalidade semanal diaria.
MAXITER = 50
M_SEASONAL = 7
auto_model = auto_arima(
    y=y_train.to_numpy(),
    X=X_train_scaled.to_numpy(),
    start_p=0,
    start_q=0,
    max_p=5,
    max_q=5,
    d=None,
    max_d=2,
    start_P=0,
    start_Q=0,
    max_P=2,
    max_Q=2,
    D=None,
    max_D=1,
    seasonal=True,
    m=M_SEASONAL,
    test='kpss',
    seasonal_test='ocsb',
    stepwise=True,
    information_criterion='aic',
    with_intercept=True,
    maxiter=MAXITER,
    error_action='ignore',
    suppress_warnings=True,
    trace=True,
    n_jobs=1
)

SARIMAX_ORDER = tuple(int(value) for value in auto_model.order)
SARIMAX_SEASONAL_ORDER = tuple(int(value) for value in auto_model.seasonal_order)

print('\nOrdens selecionadas pelo auto_arima:')
print(f'  order (p,d,q): {SARIMAX_ORDER}')
print(f'  seasonal_order (P,D,Q,m): {SARIMAX_SEASONAL_ORDER}')
print(f'  Intercepto: permitido na busca; selecionado: {auto_model.with_intercept}')
print(f'  AIC: {auto_model.aic():,.3f}')
print(f'  BIC: {auto_model.bic():,.3f}')
print(f'  Exogenas: {len(FEATURE_COLS)} | m semanal: {M_SEASONAL}')
print(f'  Convergiu na busca: {auto_model.arima_res_.mle_retvals.get("converged", "nao informado")}')
print('\nResumo do modelo selecionado:')
print(auto_model.summary())

Performing stepwise search to minimize aic
 ARIMA(0,0,0)(0,0,0)[7] intercept   : AIC=34843.891, Time=0.53 sec
 ARIMA(1,0,0)(1,0,0)[7] intercept   : AIC=34285.248, Time=11.12 sec
 ARIMA(0,0,1)(0,0,1)[7] intercept   : AIC=33666.697, Time=12.89 sec
 ARIMA(0,0,0)(0,0,0)[7]             : AIC=59567.062, Time=0.46 sec
 ARIMA(0,0,1)(0,0,0)[7] intercept   : AIC=33644.479, Time=17.56 sec
 ARIMA(0,0,1)(1,0,0)[7] intercept   : AIC=33748.295, Time=40.32 sec
 ARIMA(0,0,1)(1,0,1)[7] intercept   : AIC=33752.639, Time=48.68 sec
 ARIMA(1,0,1)(0,0,0)[7] intercept   : AIC=33879.735, Time=6.65 sec
 ARIMA(0,0,2)(0,0,0)[7] intercept   : AIC=33655.357, Time=20.41 sec
 ARIMA(1,0,0)(0,0,0)[7] intercept   : AIC=34283.479, Time=0.98 sec
 ARIMA(1,0,2)(0,0,0)[7] intercept   : AIC=33716.402, Time=7.07 sec
 ARIMA(0,0,1)(0,0,0)[7]             : AIC=inf, Time=8.60 sec

Best model:  ARIMA(0,0,1)(0,0,0)[7] intercept
Total fit time: 175.320 seconds

Ordens selecionadas pelo auto_arima:
  order (p,d,q): (0, 0, 1)
  seasona